# Data Quality Validation Notebook
This notebook tests the validation functions implemented in `src/validation/quality_checks.py` against the bronze data files.

In [1]:
import pandas as pd
import os
import sys

# Add the project root to sys.path to import from src
sys.path.append(os.path.abspath('..'))

from src.validation.quality_checks import (
    check_nulls, 
    check_duplicates, 
    check_ranges, 
    check_referential_integrity, 
    check_datatypes
)

## 1. Load Data Files

In [2]:
data_path = '../data/bronze/'
distributor_seasonality = pd.read_csv(os.path.join(data_path, 'distributor_seasonality_details.csv'))
holiday_list = pd.read_csv(os.path.join(data_path, 'holiday_list.csv'))
outlet_coordinates = pd.read_csv(os.path.join(data_path, 'outlet_coordinates.csv'))
outlet_master = pd.read_csv(os.path.join(data_path, 'outlet_master.csv'))

print("Static data files loaded successfully.")

# Loading transactions (Large file)
transactions = pd.read_csv(os.path.join(data_path, 'transactions_history_final.csv'))
print(f"Transactions loaded. Shape: {transactions.shape}")

Static data files loaded successfully.
Transactions loaded. Shape: (2376389, 7)


## 2. Validation for Outlet Master

In [3]:
print("--- Outlet Master Validation ---")
null_report = check_nulls(outlet_master, outlet_master.columns.tolist())
print("Null counts:", null_report)

id_col = 'Outlet_ID'
dup_count = check_duplicates(outlet_master, [id_col])
print(f"Number of duplicates in {id_col}: {dup_count}")

Column 'Outlet_Size' has 196 null values.


--- Outlet Master Validation ---
Null counts: {'Outlet_ID': 0, 'Outlet_Size': 196, 'Cooler_Count': 0, 'Outlet_Type': 0}
Number of duplicates in Outlet_ID: 0


## 3. Validation for Outlet Coordinates

In [ ]:
print("--- Outlet Coordinates Validation ---")
invalid_lat = check_ranges(outlet_coordinates, 'latitude', -90, 90)
print(f"Invalid Latitudes found: {len(invalid_lat)}")

invalid_lon = check_ranges(outlet_coordinates, 'longitude', -180, 180)
print(f"Invalid Longitudes found: {len(invalid_lon)}")

missing_from_master = check_referential_integrity(outlet_coordinates, outlet_master, 'outlet_id', 'Outlet_ID')
print(f"Outlets in coordinates missing from master: {len(missing_from_master)}")

KeyError: 'outlet_id'

## 4. Validation for Transactions

In [6]:
print("--- Transactions Validation ---")

# Null checks
trans_cols = ['Outlet_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value']
trans_nulls = check_nulls(transactions, trans_cols)
print("Transaction Nulls:", trans_nulls)

# Range checks
invalid_vol = check_ranges(transactions, 'Volume_Liters', 0, 10000) # Assuming 10k is a reasonable max per row
print(f"Transactions with invalid Volume_Liters (<0): {len(transactions[transactions['Volume_Liters'] < 0])}")

invalid_value = check_ranges(transactions, 'Total_Bill_Value', 0, 10000000) 
print(f"Transactions with invalid Total_Bill_Value (<0): {len(transactions[transactions['Total_Bill_Value'] < 0])}")

# Referential integrity
missing_outlets = check_referential_integrity(transactions, outlet_master, 'Outlet_ID', 'Outlet_ID')
print(f"Unique Outlets in transactions missing from master: {len(missing_outlets)}")

if len(missing_outlets) > 0:
    print("First 5 missing outlets:", missing_outlets[:5])

Column 'Volume_Liters' has 4753 values outside the range [0, 10000].
Column 'Total_Bill_Value' has 4753 values outside the range [0, 10000000].


--- Transactions Validation ---
Transaction Nulls: {'Outlet_ID': 0, 'SKU_ID': 0, 'Volume_Liters': 0, 'Total_Bill_Value': 0}
Transactions with invalid Volume_Liters (<0): 4753
Transactions with invalid Total_Bill_Value (<0): 4753
Unique Outlets in transactions missing from master: 0
